In [1]:
import ffmpeg
import subprocess
import json
from tqdm import tqdm
import time
import math 
import cv2
import numpy as np
import holoviews as hv
import panel as pn
from holoviews import streams

In [2]:
# Specify the video file path
base_dir = '
video_file = "/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m311/2025_04_07_311_19_43_52_b2/My_WebCam/2025_04_07_311_19_43_52_b2_concactenatedbehavCam00_behavCam20.mp4"
output_file = "/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m311/2025_04_07_311_19_43_52_b2/My_WebCam/2025_04_07_311_19_43_52_b2_concactenatedbehavCam00_behavCam20_rotated_30deg_padded.mp4"
#input angle to rotate video 
angleToRotate = 29.12

In [3]:
# Use ffmpeg to retrieve the metadata
try:
    # Run ffmpeg to get the file's metadata
    probe = ffmpeg.probe(video_file)
    
    # Extract the codec type for the video stream
    video_stream = next((stream for stream in probe['streams'] if stream['codec_type'] == 'video'), None)
    
    if video_stream:
        codec_name = video_stream.get('codec_name', 'Unknown')
        print(f"Video codec: {codec_name}")
    else:
        print("No video stream found.")
        
except ffmpeg.Error as e:
    print("An error occurred while reading the video file:", e)

input_file = video_file

# Get the video dimensions and duration
probe = ffmpeg.probe(input_file)
video_info = next(stream for stream in probe['streams'] if stream['codec_type'] == 'video')
width = int(video_info['width'])
height = int(video_info['height'])
duration = float(video_info['duration'])

# Calculate new dimensions to fit the rotated video
angle_rad = math.radians(angleToRotate)
new_width = int(abs(width * math.cos(angle_rad)) + abs(height * math.sin(angle_rad)))
new_height = int(abs(width * math.sin(angle_rad)) + abs(height * math.cos(angle_rad)))

# Define the ffmpeg command to add padding, rotate, and output with JPEG compression
ffmpeg_command = (
    ffmpeg.input(input_file)
    .filter('pad', new_width, new_height, (new_width - width) // 2, (new_height - height) // 2, color='0xFFFFFF')
    .filter('rotate', '30*PI/180')
    .output(output_file, vcodec='mjpeg', vsync='vfr')
    .global_args('-progress', 'pipe:1', '-nostats')  # Enable progress output
)

# Run ffmpeg as a subprocess and track progress
process = ffmpeg_command.run_async(pipe_stdout=True, pipe_stderr=True, overwrite_output=True)
pbar = tqdm(total=duration, desc="Processing Video", unit="s", dynamic_ncols=True)

# Track ffmpeg progress
for line in process.stderr:
    line = line.decode('utf-8').strip()
    if "out_time_ms" in line:
        out_time_ms = int(line.split('=')[1].strip())
        current_time = out_time_ms / 1_000_000  # Convert to seconds
        pbar.update(current_time - pbar.n)

pbar.close()
process.wait()
print("Processing complete.")


Video codec: mpeg2video


Processing Video:   0%|                           | 0/342.816667 [00:25<?, ?s/s]

Processing complete.


In [4]:
#crop video 
# Input and output file paths
input_file = "/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m311/2025_04_07_311_19_43_52_b2/My_WebCam/2025_04_07_311_19_43_52_b2_concactenatedbehavCam00_behavCam20_rotated_30deg_padded.mp4"
output_file = "/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m311/2025_04_07_311_19_43_52_b2/My_WebCam/2025_04_07_311_19_43_52_b2_concactenatedbehavCam00_behavCam20_rotated_cropped_output.avi"
# Define the four vertices of the rectangle for cropping (x1, y1), (x2, y2), (x3, y3), (x4, y4)
vertices = [(54, 373), (714, 373), (714, 404), (54, 404)]


# Get the video dimensions
probe = ffmpeg.probe(input_file)
video_info = next(stream for stream in probe['streams'] if stream['codec_type'] == 'video')
width = int(video_info['width'])
height = int(video_info['height'])

# Calculate the bounding box for cropping
x_coordinates = [v[0] for v in vertices]
y_coordinates = [v[1] for v in vertices]
x_min, x_max = max(0, min(x_coordinates)), min(width, max(x_coordinates))
y_min, y_max = max(0, min(y_coordinates)), min(height, max(y_coordinates))

# Calculate crop dimensions
crop_x = x_min
crop_y = y_min
crop_width = x_max - x_min
crop_height = y_max - y_min

# Validate crop dimensions to ensure they are within the video frame
if crop_width <= 0 or crop_height <= 0 or crop_x + crop_width > width or crop_y + crop_height > height:
    raise ValueError(f"Invalid crop dimensions: {crop_width}x{crop_height} at position ({crop_x}, {crop_y}). "
                     f"Ensure crop area is within video bounds {width}x{height}.")

# Define the ffmpeg command to crop the video
ffmpeg_command = (
    ffmpeg.input(input_file)
    .filter('crop', crop_width, crop_height, crop_x, crop_y)
    .output(output_file, vcodec='libx264', crf=23, pix_fmt='yuv420p')  # Adjust codec/compression as needed
)

# Run ffmpeg command
ffmpeg_command.run(overwrite_output=True)
print("Cropping complete.")

ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 15.0.0 (clang-1500.1.0.2.5)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --

Cropping complete.


[out#0/avi @ 0x6000015b0240] video:2239KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 22.231527%
frame=20569 fps=2121 q=31.0 Lsize=    2737KiB time=00:05:42.78 bitrate=  65.4kbits/s speed=35.3x    
[libx264 @ 0x155e061d0] frame I:83    Avg QP:25.79  size:   877
[libx264 @ 0x155e061d0] frame P:5183  Avg QP:28.11  size:   225
[libx264 @ 0x155e061d0] frame B:15303 Avg QP:32.10  size:    69
[libx264 @ 0x155e061d0] consecutive B-frames:  0.8%  0.0%  0.0% 99.2%
[libx264 @ 0x155e061d0] mb I  I16..4: 52.2%  2.8% 45.1%
[libx264 @ 0x155e061d0] mb P  I16..4:  6.3%  0.6%  1.5%  P16..4: 38.3%  5.3%  2.4%  0.0%  0.0%    skip:45.6%
[libx264 @ 0x155e061d0] mb B  I16..4:  0.6%  0.1%  0.0%  B16..8: 22.7%  1.4%  0.2%  direct: 1.0%  skip:74.0%  L0:49.3% L1:46.9% BI: 3.8%
[libx264 @ 0x155e061d0] 8x8 transform intra:6.6% inter:52.6%
[libx264 @ 0x155e061d0] coded y,uvDC,uvAC intra: 26.1% 62.3% 31.0% inter: 4.0% 11.5% 1.1%
[libx264 @ 0x155e061d0] i16 v,h,dc,p:  2% 98%  0%